In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/jp-morgan-and-chase-data/JPM.csv


In [4]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from pykalman import KalmanFilter
from sklearn.preprocessing import StandardScaler

plt.style.use('dark_background')

In [6]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [5]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/jp-morgan-and-chase-data/JPM.csv"
df = pd.read_csv(file_path)

In [7]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [8]:
# =====================================================
# 4. TERMINAL HEADER
# =====================================================
from IPython.display import display, HTML

def matrix_header(text):
    display(HTML(f"""
    <div style="
        background:#000;
        border:1px solid #00ff9c;
        padding:18px;
        margin-top:18px;
        font-family:monospace;
    ">
        <h1 style="color:#00ff9c; text-align:center;">{text}</h1>
    </div>
    """))

matrix_header("📊 Price Signal Stream")

In [9]:
# =====================================================
# 5. PRICE VISUAL (TERMINAL STYLE)
# =====================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], line=dict(color='#00ff9c')))
fig.update_layout(template='plotly_dark', title='Close Price')
fig.show()

In [10]:
# =====================================================
# 6. KALMAN FILTER TREND
# =====================================================
matrix_header("🧠 Kalman Filter Trend")

kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)
state_means, _ = kf.filter(df['close'].values)
df['kalman_trend'] = state_means

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Price'))
fig.add_trace(go.Scatter(x=df.index, y=df['kalman_trend'], name='Kalman Trend'))
fig.update_layout(template='plotly_dark')
fig.show()

In [11]:
# =====================================================
# 7. SMART MONEY FLOW (VOLUME FOOTPRINT)
# =====================================================
matrix_header("💰 Smart Money Flow")

# Proxy: price change * volume
df['flow'] = (df['close'] - df['open']) * df['volume']

fig = px.area(df, y='flow', title='Money Flow Proxy')
fig.show()

In [12]:
# =====================================================
# 8. Z-SCORE MEAN REVERSION ENGINE
# =====================================================
matrix_header("📉 Z-Score Engine")

mean = df['close'].rolling(20).mean()
std = df['close'].rolling(20).std()
df['zscore'] = (df['close'] - mean) / std

fig = px.line(df, y='zscore', title='Z-Score')
fig.show()

In [13]:
# =====================================================
# 9. PAIRS-STYLE SPREAD (SELF-NORMALIZED)
# =====================================================
matrix_header("🔗 Synthetic Spread Logic")

# simulate pairs via normalized deviation
scaler = StandardScaler()
df['norm_price'] = scaler.fit_transform(df[['close']])
df['spread'] = df['norm_price'] - df['kalman_trend']

fig = px.line(df, y='spread', title='Synthetic Spread')
fig.show()

In [14]:
# =====================================================
# 10. REGIME CLASSIFICATION
# =====================================================
matrix_header("⚡ Regime Switch")

vol = df['close'].pct_change().rolling(20).std()
df['regime'] = np.where(vol > vol.median(), 'High Vol', 'Low Vol')

fig = px.scatter(df, x=df.index, y='close', color='regime')
fig.show()

/tmp/ipykernel_55/102045709.py:6: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [15]:
# =====================================================
# 11. STRATEGY SIGNAL
# =====================================================
matrix_header("🎯 Strategy Engine")

signal = (df['zscore'] < -1) & (df['flow'] > 0)
df['signal'] = signal.astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Close'))
fig.add_trace(go.Scatter(x=df.index[df['signal']==1], y=df['close'][df['signal']==1],
                         mode='markers', name='Buy'))
fig.update_layout(template='plotly_dark')
fig.show()

In [16]:
# =====================================================
# 12. FINAL INSIGHTS
# =====================================================
matrix_header("📌 Terminal Output")

print("""
1. Kalman filter provides adaptive trend tracking.
2. Flow proxy highlights institutional pressure.
3. Z-score identifies mean reversion edges.
4. Spread logic mimics pairs trading behavior.
5. Combined signals improve timing precision.
""")

# =====================================================
# END
# =====================================================


1. Kalman filter provides adaptive trend tracking.
2. Flow proxy highlights institutional pressure.
3. Z-score identifies mean reversion edges.
4. Spread logic mimics pairs trading behavior.
5. Combined signals improve timing precision.

